# Deep Learning Notes: Optimizers in PyTorch
**Google Colab study notebook**

## Learning objectives
By the end of this notebook, you should be able to:
- Explain what an optimizer does during neural-network training.
- Understand learning rate, SGD, Momentum, RMSprop, Adam, and AdamW.
- Implement optimizers in PyTorch and compare their training behavior.
- Choose a sensible optimizer and tune its learning rate.

> **Core idea:** a loss function measures how wrong the model is; backpropagation calculates gradients; the optimizer uses those gradients to update model parameters.

## 1. What is an optimizer?

An **optimizer** is an algorithm that updates a model's trainable parameters to reduce the loss.

Typical training loop:

1. Forward pass: compute predictions.
2. Loss: compare predictions with targets.
3. `loss.backward()`: calculate gradients.
4. `optimizer.step()`: update parameters using gradients.
5. `optimizer.zero_grad()`: clear old gradients before the next batch.

### Gradient descent intuition

For parameter \(\theta\), basic gradient descent updates:

\[
\theta_{t+1}=\theta_t-\eta\nabla_\theta L(\theta_t)
\]

- \(\theta\): model parameter
- \(L\): loss function
- \(\nabla_\theta L\): gradient of the loss
- \(\eta\): learning rate

The **learning rate** controls the size of each update. Too large may cause unstable training; too small may make learning very slow.

## 2. Common optimizers at a glance

| Optimizer | Main idea | Strengths | Watch-outs |
|---|---|---|---|
| **SGD** | Uses the current mini-batch gradient | Simple; useful baseline; can generalize well | May need careful learning-rate scheduling |
| **SGD + Momentum** | Accumulates a moving direction from past gradients | Often faster and less zig-zagging | Momentum and learning rate need tuning |
| **RMSprop** | Scales updates using a moving average of squared gradients | Helpful when gradient scales vary | Results depend on hyperparameters |
| **Adam** | Combines momentum-like first moments with adaptive scaling | Often a strong starting point; usually converges quickly | Not always best for final generalization |
| **AdamW** | Adam with decoupled weight decay | Good default for many modern deep-learning tasks | Weight decay still needs tuning |

**Important:** no optimizer is best for every dataset and architecture. Compare candidates under the same data split, model, and training budget.

## 3. SGD, Momentum, RMSprop, Adam, and AdamW — intuition

### SGD
Uses the gradient from the current mini-batch. The learning rate directly scales the update.

### Momentum
Keeps a running direction based on earlier gradients. This can accelerate progress in consistent directions and reduce oscillation.

### RMSprop
Tracks the recent magnitude of squared gradients and scales each parameter's update accordingly.

### Adam
Maintains estimates of both the average gradient and average squared gradient, then uses them to adapt parameter-wise update sizes.

### AdamW
Similar to Adam, but applies weight decay separately from the gradient-based update. In PyTorch, use `torch.optim.AdamW` when you specifically want AdamW.

## 4. Import libraries
Run this cell in Google Colab or another Python environment with PyTorch installed.

In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 5. A small experiment: compare optimizers

We will train the same small neural network on a synthetic regression dataset. This keeps the example quick and lets us focus on the optimizer rather than dataset preparation.

For a fair comparison, we use:
- the same training and validation data,
- the same model architecture and initial weights,
- the same batch size and number of epochs,
- a reasonable learning rate for each optimizer.

Different optimizers may need different learning rates in real projects; this experiment is illustrative, not a universal ranking.

In [ ]:
# Create a reproducible synthetic regression dataset
rng = np.random.default_rng(SEED)
X = rng.uniform(-2, 2, size=(1600, 2)).astype(np.float32)
noise = rng.normal(0, 0.12, size=(1600, 1)).astype(np.float32)
y = (1.5 * np.sin(X[:, :1] * 1.8) + 0.7 * X[:, 1:2]**2 - 0.4 * X[:, 1:2] + noise).astype(np.float32)

# Split into train and validation sets
X_train, X_val = X[:1280], X[1280:]
y_train, y_val = y[:1280], y[1280:]

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_x = torch.from_numpy(X_val).to(device)
val_y = torch.from_numpy(y_val).to(device)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], y, s=8, alpha=0.35)
plt.xlabel("Feature 1")
plt.ylabel("Target")
plt.title("Synthetic regression data")
plt.show()

In [ ]:
class SmallRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

# Save one initial set of weights so each optimizer starts fairly.
torch.manual_seed(SEED)
base_model = SmallRegressor()
initial_state = copy.deepcopy(base_model.state_dict())

def make_optimizer(name, model, lr):
    if name == "SGD":
        return torch.optim.SGD(model.parameters(), lr=lr)
    if name == "SGD + Momentum":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    if name == "RMSprop":
        return torch.optim.RMSprop(model.parameters(), lr=lr)
    if name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr)
    if name == "AdamW":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    raise ValueError("Unknown optimizer")

## 6. Train and compare

The learning rates below are starting points for this small demonstration. If a loss diverges or barely changes, try adjusting the learning rate.

In [ ]:
optimizer_configs = {
    "SGD": 0.03,
    "SGD + Momentum": 0.03,
    "RMSprop": 0.001,
    "Adam": 0.001,
    "AdamW": 0.001,
}

EPOCHS = 30
history = {}

for name, lr in optimizer_configs.items():
    model = SmallRegressor().to(device)
    model.load_state_dict(initial_state)
    optimizer = make_optimizer(name, model, lr)
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        model.train()
        running_loss, n_seen = 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            predictions = model(xb)
            loss = criterion(predictions, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            n_seen += xb.size(0)

        train_losses.append(running_loss / n_seen)
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(val_x), val_y).item()
        val_losses.append(val_loss)

    history[name] = {"train": train_losses, "validation": val_losses}
    print(f"{name:16s} | final train MSE: {train_losses[-1]:.4f} | final val MSE: {val_losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
for name, values in history.items():
    plt.plot(values["validation"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.title("Validation loss by optimizer")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### How to interpret the graph
- Lower validation MSE is better for this regression task.
- Faster early loss reduction does not guarantee the best final validation performance.
- If training loss decreases but validation loss rises, investigate overfitting.
- Do not conclude that one optimizer is universally superior from a single experiment.

Try changing `EPOCHS`, `batch_size`, the learning rates, and `weight_decay` to see how the results change.

## 7. PyTorch optimizer syntax

These are common patterns you can adapt to your own model.

In [ ]:
# Assume model and loss_fn have already been defined.
model = SmallRegressor().to(device)

# Choose ONE optimizer at a time:
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer_momentum = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
optimizer_rmsprop = torch.optim.RMSprop(model.parameters(), lr=0.001)
optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer_adamw = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Standard mini-batch training pattern:
# optimizer.zero_grad()
# predictions = model(inputs)
# loss = loss_fn(predictions, targets)
# loss.backward()
# optimizer.step()

## 8. Learning-rate scheduling

A learning-rate scheduler changes the learning rate during training. This can help when progress slows or when you want smaller updates later in training.

Example: `StepLR` reduces the learning rate every few epochs.

In [ ]:
model = SmallRegressor().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# In a full training loop:
# for epoch in range(num_epochs):
#     train_one_epoch(...)
#     validate(...)
#     scheduler.step()
#
# For StepLR, call scheduler.step() once per epoch after training.
print("Initial learning rate:", optimizer.param_groups[0]["lr"])

## 9. Practical starting points

- **First baseline:** Adam or AdamW with learning rate `1e-3`.
- **CNN / image classification:** AdamW is a reasonable starting point; SGD with momentum is also worth testing.
- **Transformer models:** AdamW is commonly used, often with a learning-rate warm-up and schedule.
- **Small tabular neural network:** try Adam/AdamW first, then compare against SGD with momentum.
- **If loss becomes `NaN` or jumps wildly:** reduce the learning rate and check input scaling and loss calculations.
- **If learning is very slow:** verify gradients and try a larger learning rate carefully.
- **For reproducible comparisons:** keep splits, initialization, epochs, batch size, and evaluation consistent.

These are starting points, not fixed rules. Follow the recommendations for the specific model or paper you are reproducing.

## 10. Quick revision questions

1. What is the role of `optimizer.step()`?
2. Why do we call `optimizer.zero_grad()`?
3. What can happen if the learning rate is too high?
4. What extra idea does Momentum introduce compared with plain SGD?
5. How does Adam combine information from gradients?
6. What is the key distinction of AdamW?
7. Why should optimizers be compared on validation data rather than training loss alone?

### Short answers
1. Updates model parameters using their gradients.
2. PyTorch accumulates gradients by default, so old gradients should be cleared before the next batch.
3. Training may become unstable or fail to converge.
4. It uses a running direction based on past gradients.
5. It tracks moving averages of gradients and squared gradients.
6. It decouples weight decay from Adam's gradient-based update.
7. Validation performance estimates how well the model generalizes to unseen data.

## 11. References
- PyTorch Optimizers: https://pytorch.org/docs/stable/optim.html
- PyTorch Optimization Tutorial: https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html
- PyTorch Learning Rate Schedulers: https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

**Final takeaway:** the optimizer controls *how parameters are updated*; the learning rate is one of its most important hyperparameters. Start with a sensible baseline, monitor validation metrics, and tune experimentally.